## Перед началом работы скопируйте ноутбук к себе в рабочую папку, выберите датасет и укажите свое имя пользователя 

In [1]:
# pip install ipywidgets

In [2]:
import os
import json
import hashlib
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import display, clear_output
import ipywidgets as widgets
import httpx
import requests


DATASET_PATH = "/home/jovyan/msenikolaev/data_analysis/filtered_samples.txt"
# DATASET_PATH = "/home/jovyan/msenikolaev/data_analysis/concat-CQA.txt"

USERNAME = ""

assert USERNAME, "Имя пользователя должно быть заполнено!"
dataset_name = DATASET_PATH.split("/")[-1][:-4]
results_dir = f"/home/jovyan/msenikolaev/data_analysis/results/{dataset_name}/{USERNAME}"
os.makedirs(results_dir, exist_ok=True)

In [3]:
#Настройка API

JOB_ID = "lm-mpi-job-56c5855f-ed20-4f79-a2e5-1b012193572a"

# Генератор текстовых фрагментов
def text_fragment_generator(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    
    fragments = content.split('########################################################################################################################')
    
    for fragment in fragments:
        clean_fragment = fragment.strip()
        if clean_fragment:
            frag_hash = hashlib.sha256(clean_fragment.encode('utf-8')).hexdigest()
            yield clean_fragment, frag_hash

# Системный промпт
SYSTEM_PROMPT = """
Вы опытный редактор. Проанализируйте текст и дайте оценку:
0 - Текст плохой, содержит серьезные ошибки
1 - Текст приемлемый, но требует улучшений
2 - Текст отличный, соответствует всем стандартам

Ваш ответ должен включать:
1. Числовую оценку (0, 1 или 2)
2. Детальный комментарий с указанием недостатков

Верните ответ в формате JSON без backticks:
{
    "rating": число (0, 1 или 2),
    "comment": "строка с комментарием"
}
"""

# Настройка OpenAI
client = OpenAI(
    api_key="sk-giga-68bcceb1-3038-47ab-b4c5-be88e60cb69e-pretrain-team",
    base_url="https://chatwm.ru/f47db70ec76d1d333be922d9587fe825/openai/v1",
    timeout=100,
    http_client=httpx.Client(verify=False),
)

# 3. Функция запроса к OpenAI
def get_openai_feedback(text, frag_id):
    """Запрос оценки текста к OpenAI"""
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text}
            ],
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        # print(f"Ошибка при запросе к OpenAI: {str(e)}")
        return {"rating": -1, "comment": f"Ошибка: {str(e)}"}
    
def get_gigachat_feedback(text, frag_id):
    """Запрос оценки текста к GigaChat"""
    try:
        response = request(text)
        return json.loads(response["choices"][0]["message"]["content"])
    except Exception as e:
        print(f"Ошибка при запросе к GigaChat: {str(e)}")
        return {"rating": -1, "comment": f"Ошибка: {str(e)}"}
    
def request(text):
    url = f"http://{JOB_ID}-mpimaster-0.{JOB_ID}:8081/chat"  
    headers = {
        'Content-Type': 'application/json'
    }
    data = {
        "messages":[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text}
            ],
        "model": "llm05-30-2025_step1"
    }

    response = requests.post(url, headers=headers, json=data)

    if response.status_code == 200:
        result = response.json()
        return result
    else:
        print("Ошибка:", response.status_code, response.text)

def anthropic_request(text):
    client = Anthropic(
        api_key="sk-giga-68bcceb1-3038-47ab-b4c5-be88e60cb69e-pretrain-team",
        base_url="https://chatwm.ru/f47db70ec76d1d333be922d9587fe825/anthropic",
        timeout=100,
        http_client=httpx.Client(verify=False),
    )
    response = client.messages.create(
        model="claude-3-5-sonnet-latest",
        system=SYSTEM_PROMPT,
        messages=[
                {"role": "user", "content": text}
            ],
        max_tokens=2048,  # Обязательный параметр для Anthropic
    )
    
    result = response.to_dict()
    return result
    

def get_anthropic_feedback(text):
    """Запрос оценки текста к Anthropic"""
    try:
        response = anthropic_request(text)
        return json.loads(response['content'][0]['text'])
    except Exception as e:
        print(f"Ошибка при запросе к Anthropic: {str(e)}")
        return {"rating": -1, "comment": f"Ошибка: {str(e)}"}

In [4]:
#UI элементы

class ToggleResponseWidget:
    def __init__(self, model_name):
        self.output = widgets.HTML(
            value=f'<div class="text-container">Тут будет ответ {model_name}</div>',
            layout={'width': '90%', 
                    'height': '300px', 
                    'overflow': 'auto',
                    'display': 'none' 
            },
            style={
                'font_family': 'Georgia',  # Моноширинный шрифт для текста
                'background': '#F5F5F5',     # Светлый фон
                'padding': '50px',           # Отступы внутри блока
                'font_size': '18px'
            }
        )
        self.model_name = model_name
        self.toggle_btn = widgets.Button(
            description=f'Показать ответ {self.model_name}',
            icon='chevron-down',
            layout={'width': '200px'}
        )
        # self.toggle_btn.style.button_color = '#f0f0f0'
        
        self.toggle_btn.on_click(self.toggle_output)
        
        # Контейнер
        self.widget = widgets.VBox([
            self.toggle_btn,
            self.output
        ])
    
    def toggle_output(self, btn):
        """Переключает видимость контента"""
        if self.output.layout.display == 'none':
            self.output.layout.display = 'block'
            self.toggle_btn.description = f'Скрыть ответ {self.model_name}'
            self.toggle_btn.icon = 'chevron-up'
        else:
            self.output.layout.display = 'none'
            self.toggle_btn.description = f'Показать ответ {self.model_name}'
            self.toggle_btn.icon = 'chevron-down'
    
    def update_response(self, text):
        """Обновляет текст ответа"""
        escaped_text = (text.replace('&', '&amp;')
                   .replace('<', '&lt;')
                   .replace('>', '&gt;')
                   .replace('\n', '<br>'))
        self.output.value = f'<div class="text-container">{escaped_text}</div>'
    
    def show(self):
        """Отображает виджет"""
        display(self.widget)

def beauty_response(model_feedback):
    response = ""
    response += f'Оценка: {model_feedback.get("rating", -1) if model_feedback else None}\n\n'
    response += f'Комментарий: {model_feedback.get("comment", "") if model_feedback else None}\n'
    return response

text_display = widgets.HTML(
    value='<div class="text-container">Текст появится здесь...</div>',
    layout={'width': '90%', 
            'height': '450px', 
            'overflow': 'auto',
    },
    style={
        'font_family': 'Georgia',  # Моноширинный шрифт для текста
        'background': '#F5F5F5',     # Светлый фон
        'padding': '50px',           # Отступы внутри блока
        'font_size': '18px'
    }
)

# Функция обновления текста
def update_text(text):
    escaped_text = (text.replace('&', '&amp;')
                   .replace('<', '&lt;')
                   .replace('>', '&gt;')
                   .replace('\n', '<br>'))
    text_display.value = f'<div class="text-container">{escaped_text}</div>'

status_display = widgets.Output(layout={'width': '100%', 'height': '60px'})

rating_buttons = widgets.ToggleButtons(
    options=[0, 1, 2],
    description="0 (Плохо), 1 (Нормально), 2 (Отлично)",
    button_style='primary',  # primary 'success', 'warning', 'danger' не работают в ToggleButtons
    style={
        'button_width': '300px',
        },
    
)

comment_input = widgets.Textarea(
    placeholder='Введите ваш комментарий здесь...',
    layout={'width': '90%', 'height': '100px'}
)

save_button = widgets.Button(
    description="СОХРАНИТЬ ОЦЕНКУ",
    button_style='success',
    layout={'width': '300px', 'height': '50px'},
    style={'font_weight': 'bold'}
)

giga_response_widget = ToggleResponseWidget('GigaChat')
gpt_response_widget = ToggleResponseWidget('ChatGPT')
anthropic_response_widget = ToggleResponseWidget('Anthropic')

current_fragment = None
current_frag_id = None
gpt_feedback = None
giga_feedback = None
anthropic_feedback = None
generator = None

In [5]:
#Логика работы

def init_generator(data_path):
    global generator
    try:
        generator = text_fragment_generator(data_path)
        load_next_fragment()
    except FileNotFoundError:
        with status_display:
            print(f"Ошибка: Файл {data_path} не найден!")

def load_next_fragment():
    global current_fragment, current_frag_id, giga_feedback, gpt_feedback
    
    try:
        while True:
            current_fragment, current_frag_id = next(generator)
            result_file = os.path.join(results_dir, f"{current_frag_id}.json")
            
            if not os.path.exists(result_file):
                break
        
        # Запускаем запрос к модели
        with status_display:
            clear_output()
            print("Отправка запроса к модели...")


        update_text(current_fragment)

        rating_buttons.value = None
        comment_input.value = ''

        clean_model_feedback()

        save_model_feedback(current_fragment, current_frag_id)
        
        giga_response_widget.update_response(beauty_response(giga_feedback))
        gpt_response_widget.update_response(beauty_response(gpt_feedback))
        anthropic_response_widget.update_response(beauty_response(anthropic_feedback))

        with status_display:
            clear_output()
            print("Ожидание оценки пользователя")

        
    except StopIteration:
        update_text("Все фрагменты обработаны!")
        with status_display:
            print("Работа завершена")
        save_button.disabled = True

def clean_model_feedback():
    global giga_feedback, gpt_feedback, anthropic_feedback
    gpt_feedback = None
    giga_feedback = None
    anthropic_feedback = None

def save_model_feedback(text, frag_id):
    global giga_feedback, gpt_feedback, anthropic_feedback
    gpt_feedback = get_openai_feedback(text, frag_id)
    giga_feedback = get_gigachat_feedback(text, frag_id)
    anthropic_feedback = get_anthropic_feedback(text)

def on_save_clicked(b):
    if rating_buttons.value is None:
        with status_display:
            clear_output()
            print("Ошибка: Выберите оценку перед сохранением!")
        return
    
    if comment_input.value == '' and rating_buttons.value < 2:
        with status_display:
            clear_output()
            print("Ошибка: Напишите комментарий перед сохранением!")
        return
    
    if not current_fragment:
        return
    

    result = {
        "id": current_frag_id,
        "text": current_fragment,
        "human_rating": rating_buttons.value,
        "human_comment": comment_input.value,
        "chatGPT_rating": gpt_feedback.get("rating", -1) if gpt_feedback else None,
        "chatGPT_comment": gpt_feedback.get("comment", "") if gpt_feedback else None,
        "GigaChat_rating": giga_feedback.get("rating", -1) if giga_feedback else None,
        "GigaChat_comment": giga_feedback.get("comment", "") if giga_feedback else None,
        "Anthropic_rating": anthropic_feedback.get("rating", -1) if anthropic_feedback else None,
        "Anthropic_comment": anthropic_feedback.get("comment", "") if anthropic_feedback else None
    }
    
    # Сохраняем результат
    result_file = os.path.join(results_dir, f"{current_frag_id}.json")
    with open(result_file, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    with status_display:
        print(f"Сохранено в {result_file}")
    
    # Загружаем следующий фрагмент
    load_next_fragment()
    
save_button.on_click(on_save_clicked)

In [6]:
# Компоновка интерфейса
controls = widgets.VBox([
    widgets.Label("Оценка:"),
    rating_buttons,
    widgets.Label("Ваш комментарий:"),
    comment_input,
    widgets.HBox([save_button], layout={'justify_content': 'center'})
])

dashboard = widgets.VBox([
    widgets.Label("Текущий текст:"),
    text_display,
    giga_response_widget.widget,
    anthropic_response_widget.widget,
    gpt_response_widget.widget,
    widgets.Label("Статус:"),
    status_display,
    controls
])


In [ ]:
# Запуск системы
display(dashboard)
init_generator(DATASET_PATH)